# Credit risk & decisioningReads a JSON export from the admin console (`/lending/risk` → **JSON**). Nothing here connects to adatabase or a service: the export is the audit boundary, and it carries the policy bundle thatproduced the decisions, so every threshold drawn below is the bank's own rule, not a constanttyped into this notebook.**What the data can and cannot say.** The decision fields are the ADR-0213 evidence the enginepinned (outcome, reason codes, matched rule ids, table versions, input hash). There is no scoreand no probability — the engine does not produce one. The IFRS 9 figures come from the scheduledprovisioning cycle; the PD/LGD behind them is a flat placeholder set (see the model version cell),so an ECL here is arithmetic on assumed parameters, not a calibrated loss estimate.

In [ ]:
import jsonfrom pathlib import Pathimport pandas as pdimport matplotlib.pyplot as plt# The file the console downloaded. Point EXPORT at your own download.EXPORT = Path("credit-decisions.json")payload = json.loads(EXPORT.read_text())policy = payload.get("policy")decisions = pd.json_normalize(payload.get("decisions", []))portfolio = pd.json_normalize(payload.get("portfolio", []))print(f"{len(decisions)} decisions, {len(portfolio)} loans")print("policy:", (policy or {}).get("source"), "code-seeded:", (policy or {}).get("codeSeeded"))

## 1. What the engine decidesOutcome mix over the exported set. Remember this is the export, not the book: the console's owntiles take book-wide totals from a database aggregate, and the export is capped.

In [ ]:
outcomes = decisions["engineOutcome"].value_counts()display(outcomes.rename("count").to_frame().assign(share=lambda d: d["count"] / d["count"].sum()))ax = outcomes.reindex(["APPROVE", "REFER", "DECLINE"]).fillna(0).plot.bar(    color=["#22c55e", "#f59e0b", "#ef4444"], rot=0, figsize=(6, 3))ax.set_title("Engine outcomes (exported sample)")ax.set_ylabel("applications")plt.tight_layout()

## 2. Why — reason codes and the rule that fired`reasons` is the adverse-action contract: a machine-readable code plus the id of the rule thatproduced it. A REFER with `INPUT_MISSING` is a data problem, not a credit judgement, and the twoshould never be read as one number.

In [ ]:
reasons = decisions.explode("reasons").dropna(subset=["reasons"])if len(reasons):    reasons = pd.concat([        reasons.drop(columns=["reasons"]).reset_index(drop=True),        pd.json_normalize(reasons["reasons"]).reset_index(drop=True),    ], axis=1)    pareto = (reasons.groupby(["code", "ruleId"], dropna=False)              .size().sort_values(ascending=False).rename("count").to_frame())    display(pareto.head(15))else:    print("No reasons in this export: every decision was a clean approve.")

## 3. Against which policyThe tables the engine evaluated, in evaluation order, with how often each rule matched in thisexport. A rule with zero hits is either redundant or starved of its input — worth asking which.

In [ ]:
rules = []for table in (policy or {}).get("tables", []):    for rule in table["rules"]:        rules.append({            "kind": table["kind"], "table": table["name"], "version": table["version"],            "rule": rule["id"], "attribute": rule["attribute"], "operator": rule["operator"],            "threshold": rule["threshold"], "values": ",".join(rule["values"]), "band": rule["band"],        })rules = pd.DataFrame(rules)hits = decisions.explode("matchedRuleIds")["matchedRuleIds"].value_counts()rules["hits"] = rules["rule"].map(hits).fillna(0).astype(int)order = {"EXCLUSION": 0, "ELIGIBILITY": 1, "AFFORDABILITY": 2, "PRICING_BAND": 3}display(rules.sort_values(by=["kind", "rule"], key=lambda c: c.map(order).fillna(99) if c.name == "kind" else c))

## 4. Affordability against the bank's own thresholdsThe lines are read from the policy above. Two DSTI definitions are plotted: the one the engineevaluates (new installment / verified income) and the total-debt-service one (adding theapplicant's existing monthly debt service), which is the CNB/EBA definition. The gap between themis the part of the borrower's obligations the current policy does not see.

In [ ]:
def limit(kind: str, attribute: str):    if rules.empty:        return None    row = rules[(rules["kind"] == kind) & (rules["attribute"] == attribute) & rules["threshold"].notna()]    return None if row.empty else float(row.iloc[0]["threshold"])dsti_limit, dti_limit = limit("AFFORDABILITY", "DSTI"), limit("AFFORDABILITY", "DTI")aff = decisions.dropna(subset=["affordability.dsti"])fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)colour = {"APPROVE": "#22c55e", "REFER": "#f59e0b", "DECLINE": "#ef4444"}for ax, column, title in [    (axes[0], "affordability.dsti", "DSTI as the engine reads it"),    (axes[1], "affordability.dstiIncludingExistingDebt", "DSTI incl. existing debt service"),]:    for outcome, group in aff.groupby("engineOutcome"):        ax.scatter(group[column], group["affordability.dti"], s=22,                   c=colour.get(outcome, "#94a3b8"), label=outcome, alpha=0.8)    if dsti_limit is not None:        ax.axvline(dsti_limit, ls="--", c="#ef4444")    if dti_limit is not None:        ax.axhline(dti_limit, ls="--", c="#ef4444")    ax.set_xlabel("DSTI"); ax.set_title(title)axes[0].set_ylabel("DTI"); axes[0].legend(fontsize=8)plt.tight_layout()if dsti_limit is not None and len(aff):    over = (aff["affordability.dstiIncludingExistingDebt"] > dsti_limit).sum()    within = (aff["affordability.dsti"] <= dsti_limit).sum()    print(f"{over} of {len(aff)} applications breach the DSTI limit ONLY once existing debt service "          f"is counted; {within} pass the limit the engine applies.")

## 5. Engine versus humanWhere the four-eyes checker went the other way. A high override rate is not automatically aproblem — it can be a policy that is too tight — but it is always a question for the riskcommittee, and it is the number a model-risk review asks for first.

In [ ]:
APPROVED = {"OFFERED", "AWAITING_SIGNATURE", "SIGNED", "REFLECTION_PERIOD", "READY_TO_DISBURSE", "DISBURSED"}LAPSED = {"WITHDRAWN", "EXPIRED"}def disposition(status: str) -> str:    if status in APPROVED:        return "approved"    if status == "DECLINED":        return "declined"    if status in LAPSED:        return "lapsed"    return "in flight"decisions["disposition"] = decisions["status"].map(disposition)matrix = pd.crosstab(decisions["engineOutcome"], decisions["disposition"])display(matrix)disposed = matrix.reindex(columns=["approved", "declined"]).fillna(0)overridden = disposed.get("declined", 0).get("APPROVE", 0) + disposed.get("approved", 0).get("DECLINE", 0)total = disposed.to_numpy().sum()print("override rate:", "n/a (nothing disposed yet)" if total == 0 else f"{overridden / total:.1%}")

## 6. The book — IFRS 9 staging, ECL coverage, vintageMoney is per currency. Anything summed across currencies below is grouped first, because a singlefigure adding CZK to EUR looks authoritative and means nothing.

In [ ]:
if portfolio.empty:    print("No loans in this export.")else:    assessed = portfolio.dropna(subset=["assessment.stage"])    print(f"{len(assessed)} of {len(portfolio)} loans have a provisioning record; "          f"PD/LGD model versions: {sorted(assessed['assessment.modelVersion'].unique())}")    by_stage = (assessed.groupby(["currency", "assessment.stage"])                .agg(loans=("loanId", "size"),                     outstanding=("assessment.outstandingBalance", "sum"),                     ecl=("assessment.expectedCreditLoss", "sum")))    by_stage["coverage"] = by_stage["ecl"] / by_stage["outstanding"]    display(by_stage)    buckets = (assessed.groupby(["currency", "assessment.bucket"])               .agg(loans=("loanId", "size"), outstanding=("assessment.outstandingBalance", "sum")))    display(buckets)

In [ ]:
if not portfolio.empty:    portfolio["vintage"] = pd.to_datetime(portfolio["disbursedAt"]).dt.to_period("M").astype(str)    stage = portfolio.get("assessment.stage")    vintage = pd.crosstab(portfolio["vintage"], stage.fillna("NOT ASSESSED") if stage is not None else "NOT ASSESSED")    display(vintage)    ax = vintage.plot.bar(stacked=True, figsize=(8, 3.5), rot=0,                          color={"STAGE_1": "#6366f1", "STAGE_2": "#f59e0b", "STAGE_3": "#ef4444",                                 "NOT ASSESSED": "#cbd5e1"})    ax.set_title("Vintage: disbursement month × current IFRS 9 stage")    ax.set_ylabel("loans")    plt.tight_layout()

## 7. Read this before quoting any number above- **The bureau port is a no-op.** `CUSTOMER_TYPE` is `STANDARD` for every applicant, so the  exclusion table cannot fire. A decline rate near zero is a statement about the missing feed, not  about the applicants.- **PD/LGD are flat placeholders** (`noop-flat-v1` unless the model version above says otherwise).  ECL and coverage are arithmetic on assumed parameters.- **The policy may be code-seeded** (`codeSeeded: true`), meaning the tables can only change through  a reviewed commit and a release — there is no four-eyes activation for them yet.- **The export is capped.** Rates computed here are of the exported sample; book-wide totals come  from the console's database aggregate.